# 📚 Scrape Book Data from Ookbee Buffet
## สำหรับระบบแนะนำหนังสือ Greed Route

### วัตถุประสงค์
- ดึงข้อมูลหนังสือจาก Ookbee Buffet (buffet.ookbee.com) แยกตามหมวดหมู่
- แปลงข้อมูลเป็น JSON สำหรับใช้ใน Web App
- บันทึกไฟล์ข้อมูลไปที่ `../data/`

### แหล่งข้อมูล
- **Local HTML**: ไฟล์ HTML ที่บันทึกจาก ookbee (ใน `../doc/web_ookbee/`)
- **Live scraping**: ดึงข้อมูลจาก `buffet.ookbee.com` โดยตรง

## 1. Import Required Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import unquote

# Paths
BASE_DIR = Path(r"C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys")
DATA_DIR = BASE_DIR / "data"
DOC_DIR = BASE_DIR / "doc" / "web_ookbee"
HTML_DIR = BASE_DIR / "html"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data output: {DATA_DIR}")
print(f"Local HTML source: {DOC_DIR}")
print(f"HTML app output: {HTML_DIR}")

Data output: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data
Local HTML source: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\doc\web_ookbee
HTML app output: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\html


## 2. Configure Scraping Parameters and Genre Categories

กำหนดหมวดหมู่หนังสือ 20 หมวดที่ใช้ในระบบ Greed Route พร้อม URL สำหรับดึงข้อมูลจาก Ookbee Buffet

In [2]:
OOKBEE_BASE = "https://buffet.ookbee.com"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "th-TH,th;q=0.9,en;q=0.8",
}

DELAY = 2  # seconds between requests (polite scraping)

# 20 genre categories mapping: id -> (thai_name, english_name, ookbee_category_url_path)
GENRE_CATEGORIES = [
    {"id": "business",   "th": "ธุรกิจ การเงิน และการลงทุน",       "en": "Business, Finance & Investment",
     "desc_th": "ครอบคลุมกลยุทธ์ธุรกิจและเทคนิคการลงทุน", "desc_en": "Business strategies and investment techniques",
     "ookbee_path": "/catalog/book/category/ธุรกิจ การเงิน และการลงทุน"},
    {"id": "psychology", "th": "จิตวิทยาและการพัฒนาตนเอง",         "en": "Psychology & Self-Development",
     "desc_th": "หมวดหมู่ยอดฮิตสำหรับการเพิ่มศักยภาพให้ตัวเอง", "desc_en": "Popular category for self-improvement",
     "ookbee_path": "/catalog/book/category/จิตวิทยาและการพัฒนาตนเอง"},
    {"id": "romance",    "th": "นิยายโรแมนซ์",                      "en": "Romance",
     "desc_th": "เรื่องราวความรักที่ได้รับความนิยมสูงในทุกแพลตฟอร์ม", "desc_en": "Love stories popular across all platforms",
     "ookbee_path": None},
    {"id": "mystery",    "th": "สืบสวนสอบสวนและระทึกขวัญ",          "en": "Mystery & Thriller",
     "desc_th": "ปริศนาและการลุ้นระทึกที่น่าติดตาม", "desc_en": "Mysteries and thrilling suspense",
     "ookbee_path": None},
    {"id": "fantasy",    "th": "แฟนตาซี",                           "en": "Fantasy",
     "desc_th": "การผจญภัยในโลกแห่งจินตนาการและเวทมนตร์", "desc_en": "Adventures in worlds of imagination and magic",
     "ookbee_path": None},
    {"id": "scifi",      "th": "วิทยาศาสตร์",                       "en": "Science Fiction",
     "desc_th": "เรื่องราวอนาคต เทคโนโลยี และอวกาศ", "desc_en": "Future stories, technology, and space",
     "ookbee_path": None},
    {"id": "history",    "th": "ประวัติศาสตร์",                      "en": "History",
     "desc_th": "เจาะลึกเหตุการณ์สำคัญและการเมืองในอดีต", "desc_en": "Deep dive into important events and politics",
     "ookbee_path": None},
    {"id": "biography",  "th": "ชีวประวัติและบันทึกความทรงจำ",       "en": "Biography & Memoir",
     "desc_th": "เรียนรู้จากชีวิตจริงของบุคคลสำคัญ", "desc_en": "Learn from the real lives of notable people",
     "ookbee_path": None},
    {"id": "tech",       "th": "คอมพิวเตอร์และเทคโนโลยี",           "en": "Computer & Technology",
     "desc_th": "อัปเดตเทรนด์ AI, Coding และนวัตกรรม", "desc_en": "AI, Coding trends and innovation",
     "ookbee_path": "/catalog/book/category/คอมพิวเตอร์และเทคโนโลยี"},
    {"id": "comics",     "th": "การ์ตูนและมังงะ",                    "en": "Comics & Manga",
     "desc_th": "ทั้งมังงะญี่ปุ่นและการ์ตูนไทยยอดนิยม", "desc_en": "Japanese manga and popular Thai comics",
     "ookbee_path": "/catalog/book/category/การ์ตูนจากญี่ปุ่น"},
    {"id": "ya",         "th": "วรรณกรรมเยาวชน",                    "en": "Young Adult",
     "desc_th": "เรื่องราวการเติบโตและผจญภัยที่อ่านได้ทุกวัย", "desc_en": "Coming-of-age stories for all ages",
     "ookbee_path": "/catalog/book/category/การ์ตูนเด็กและเยาวชน"},
    {"id": "food",       "th": "อาหารและเครื่องดื่ม",                "en": "Cookbooks & Food",
     "desc_th": "สูตรอาหารและศิลปะการทำอาหาร", "desc_en": "Recipes and the art of cooking",
     "ookbee_path": "/catalog/book/category/อาหารและเครื่องดื่ม"},
    {"id": "travel",     "th": "ท่องเที่ยว",                         "en": "Travel",
     "desc_th": "แรงบันดาลใจในการเดินทางและคู่มือท่องเที่ยว", "desc_en": "Travel inspiration and guides",
     "ookbee_path": "/catalog/book/category/ท่องเที่ยว"},
    {"id": "religion",   "th": "ศาสนาและความเชื่อ",                  "en": "Religion & Beliefs",
     "desc_th": "ปรัชญาการใช้ชีวิตและจิตวิญญาณ", "desc_en": "Life philosophy and spirituality",
     "ookbee_path": "/catalog/book/category/ศาสนาและความเชื่อ"},
    {"id": "health",     "th": "สุขภาพและความงาม",                   "en": "Health & Wellness",
     "desc_th": "การดูแลร่างกายและสุขภาพจิต", "desc_en": "Body care and mental health",
     "ookbee_path": None},
    {"id": "art",        "th": "ศิลปะและการออกแบบ",                  "en": "Art & Design",
     "desc_th": "ความคิดสร้างสรรค์และสถาปัตยกรรม", "desc_en": "Creativity and architecture",
     "ookbee_path": None},
    {"id": "general",    "th": "ความรู้ทั่วไป",                      "en": "General Knowledge",
     "desc_th": "สารคดีและเรื่องน่ารู้อื่นๆ", "desc_en": "Documentaries and interesting facts",
     "ookbee_path": "/catalog/book/category/ความรู้ทั่วไป"},
    {"id": "language",   "th": "ภาษาและการศึกษา",                    "en": "Language & Education",
     "desc_th": "การฝึกทักษะภาษาและตำราเรียน", "desc_en": "Language skills and textbooks",
     "ookbee_path": None},
    {"id": "family",     "th": "ครอบครัวและเด็ก",                    "en": "Family & Parenting",
     "desc_th": "คู่มือการเลี้ยงลูกและความสัมพันธ์ในครอบครัว", "desc_en": "Parenting guides and family relationships",
     "ookbee_path": "/catalog/book/category/ครอบครัว/แม่และเด็ก"},
    {"id": "classics",   "th": "วรรณกรรมคลาสสิก",                   "en": "Classics",
     "desc_th": "หนังสือทรงคุณค่าที่ผ่านกาลเวลา", "desc_en": "Timeless valuable books",
     "ookbee_path": None},
]

print(f"Defined {len(GENRE_CATEGORIES)} genre categories")
print(f"Categories with ookbee URL: {sum(1 for g in GENRE_CATEGORIES if g['ookbee_path'])}")

Defined 20 genre categories
Categories with ookbee URL: 10


## 3. Parse Saved Local HTML File (Offline Scraping)

ดึงข้อมูลหนังสือจากไฟล์ HTML ที่บันทึกไว้จาก Ookbee (หมวดจิตวิทยาและการพัฒนาตนเอง)

In [3]:
def parse_ookbee_html(html_path, genre_id="psychology"):
    """Parse a saved Ookbee category HTML file and extract book data."""
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    
    books = []
    grid = soup.select("#catalog-grid a")
    
    for item in grid:
        href = item.get("href", "")
        book_id = href.split("/book/")[-1] if "/book/" in href else ""
        
        title_el = item.select_one("p.text-single-line")
        title = title_el.get("title", title_el.text.strip()) if title_el else ""
        
        img_el = item.select_one("img.full-width")
        cover_url = img_el.get("src", "") if img_el else ""
        
        if title:
            books.append({
                "book_id": book_id,
                "title_th": title,
                "title_en": title,  # Ookbee titles are often mixed Thai/English
                "author": "",       # Not available in grid view
                "cover_url": cover_url,
                "genre": genre_id,
                "rating": round(3.5 + (hash(title) % 15) / 10, 1),  # Simulated rating 3.5-5.0
                "source": "ookbee_local"
            })
    
    return books

# Parse the saved psychology category page
local_html = DOC_DIR / "ex01_จิตวิทยาและการพัฒนาตนเอง.html"
if local_html.exists():
    local_books = parse_ookbee_html(local_html, "psychology")
    print(f"✅ Parsed {len(local_books)} books from local HTML")
    for b in local_books[:5]:
        print(f"   📕 {b['title_th']}")
else:
    local_books = []
    print(f"❌ Local HTML not found: {local_html}")

✅ Parsed 20 books from local HTML
   📕 LEAN คิดแบบผู้นำ ทำแบบคนสำเร็จ
   📕 สอนคิดตามแนวทาง Thinking School
   📕 คนแบบไหนอยู่ด้วยแล้วสบายใจ คนแบบไหนอยู่ใกล้แล้วเพลีย
   📕 COACHING เทคนิคกระตุ้นทีมจนสำเร็จ
   📕 แสงอาทิตย์ไม่คิดตังค์


## 4. Scrape Book Data from Ookbee (Live)

ดึงข้อมูลหนังสือจาก buffet.ookbee.com แยกตามหมวดหมู่ที่มี URL (polite scraping with delay)

> ⚠️ ตั้งค่า `LIVE_SCRAPE = True` เพื่อเปิดการ scrape จากเว็บไซต์จริง (ใช้เวลาประมาณ 2 วินาทีต่อหมวด)

In [4]:
LIVE_SCRAPE = False  # Set to True to scrape from ookbee.com

def scrape_ookbee_category(category_path, genre_id, max_books=20):
    """Scrape books from a single Ookbee category page."""
    url = OOKBEE_BASE + category_path
    print(f"  🌐 Scraping: {url}")
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        resp.encoding = "utf-8"
    except requests.RequestException as e:
        print(f"  ❌ Error: {e}")
        return []
    
    soup = BeautifulSoup(resp.text, "html.parser")
    books = []
    
    for item in soup.select("#catalog-grid a")[:max_books]:
        href = item.get("href", "")
        book_id = href.split("/book/")[-1] if "/book/" in href else ""
        
        title_el = item.select_one("p.text-single-line")
        title = title_el.get("title", title_el.text.strip()) if title_el else ""
        
        img_el = item.select_one("img.full-width")
        cover_url = img_el.get("src", "") if img_el else ""
        
        if title:
            books.append({
                "book_id": book_id,
                "title_th": title,
                "title_en": title,
                "author": "",
                "cover_url": cover_url,
                "genre": genre_id,
                "rating": round(3.5 + (hash(title) % 15) / 10, 1),
                "source": "ookbee_live"
            })
    
    return books

# Scrape categories that have ookbee URLs
scraped_books = []
if LIVE_SCRAPE:
    for genre in GENRE_CATEGORIES:
        if genre["ookbee_path"]:
            books = scrape_ookbee_category(genre["ookbee_path"], genre["id"])
            scraped_books.extend(books)
            print(f"  ✅ {genre['th']}: {len(books)} books")
            time.sleep(DELAY)
    print(f"\n📊 Total scraped: {len(scraped_books)} books")
else:
    print("ℹ️ Live scraping disabled. Set LIVE_SCRAPE = True to scrape from ookbee.com")
    print("   Using local HTML + generated sample data instead.")

ℹ️ Live scraping disabled. Set LIVE_SCRAPE = True to scrape from ookbee.com
   Using local HTML + generated sample data instead.


## 5. Generate Sample Data for Categories without Ookbee URLs

สำหรับหมวดหมู่ที่ไม่มี URL ใน Ookbee จะสร้างข้อมูลตัวอย่างเพื่อให้ครบทุกหมวด

In [5]:
# Sample books data - well-known titles for each genre
SAMPLE_BOOKS = {
    "business": [
        {"title_th": "Rich Dad Poor Dad", "title_en": "Rich Dad Poor Dad", "author": "Robert Kiyosaki", "rating": 4.2},
        {"title_th": "The Psychology of Money", "title_en": "The Psychology of Money", "author": "Morgan Housel", "rating": 4.6},
        {"title_th": "Think and Grow Rich", "title_en": "Think and Grow Rich", "author": "Napoleon Hill", "rating": 4.3},
        {"title_th": "The Intelligent Investor", "title_en": "The Intelligent Investor", "author": "Benjamin Graham", "rating": 4.4},
        {"title_th": "Zero to One", "title_en": "Zero to One", "author": "Peter Thiel", "rating": 4.2},
    ],
    "psychology": [
        {"title_th": "Atomic Habits", "title_en": "Atomic Habits", "author": "James Clear", "rating": 4.8},
        {"title_th": "Think Again", "title_en": "Think Again", "author": "Adam Grant", "rating": 4.4},
        {"title_th": "Mindset", "title_en": "Mindset", "author": "Carol S. Dweck", "rating": 4.3},
        {"title_th": "Thinking, Fast and Slow", "title_en": "Thinking, Fast and Slow", "author": "Daniel Kahneman", "rating": 4.5},
        {"title_th": "คณิตศาสตร์ของความรัก", "title_en": "The Mathematics of Love", "author": "Hannah Fry", "rating": 4.1},
    ],
    "romance": [
        {"title_th": "เพราะเราคู่กัน", "title_en": "2gether", "author": "JittiRain", "rating": 4.3},
        {"title_th": "พี่ชายที่รัก", "title_en": "Dear Brother", "author": "สะมะเรีย", "rating": 4.1},
        {"title_th": "บุพเพสันนิวาส", "title_en": "Love Destiny", "author": "รอมแพง", "rating": 4.5},
        {"title_th": "The Notebook", "title_en": "The Notebook", "author": "Nicholas Sparks", "rating": 4.2},
        {"title_th": "ลิขิตรักข้ามดวงดาว", "title_en": "Star-Crossed Love", "author": "ณารา", "rating": 4.0},
    ],
    "mystery": [
        {"title_th": "Gone Girl", "title_en": "Gone Girl", "author": "Gillian Flynn", "rating": 4.1},
        {"title_th": "The Girl with the Dragon Tattoo", "title_en": "The Girl with the Dragon Tattoo", "author": "Stieg Larsson", "rating": 4.1},
        {"title_th": "And Then There Were None", "title_en": "And Then There Were None", "author": "Agatha Christie", "rating": 4.3},
        {"title_th": "The Da Vinci Code", "title_en": "The Da Vinci Code", "author": "Dan Brown", "rating": 3.9},
        {"title_th": "คดีฆาตกรรมบนรถด่วน", "title_en": "Murder on the Orient Express", "author": "Agatha Christie", "rating": 4.2},
    ],
    "fantasy": [
        {"title_th": "The Name of the Wind", "title_en": "The Name of the Wind", "author": "Patrick Rothfuss", "rating": 4.6},
        {"title_th": "The Midnight Library", "title_en": "The Midnight Library", "author": "Matt Haig", "rating": 4.2},
        {"title_th": "Harry Potter and the Sorcerer's Stone", "title_en": "Harry Potter and the Sorcerer's Stone", "author": "J.K. Rowling", "rating": 4.8},
        {"title_th": "The Hobbit", "title_en": "The Hobbit", "author": "J.R.R. Tolkien", "rating": 4.3},
        {"title_th": "Eragon", "title_en": "Eragon", "author": "Christopher Paolini", "rating": 4.0},
    ],
    "scifi": [
        {"title_th": "Dune", "title_en": "Dune", "author": "Frank Herbert", "rating": 4.5},
        {"title_th": "Project Hail Mary", "title_en": "Project Hail Mary", "author": "Andy Weir", "rating": 4.7},
        {"title_th": "Ender's Game", "title_en": "Ender's Game", "author": "Orson Scott Card", "rating": 4.3},
        {"title_th": "The Martian", "title_en": "The Martian", "author": "Andy Weir", "rating": 4.5},
        {"title_th": "Neuromancer", "title_en": "Neuromancer", "author": "William Gibson", "rating": 3.9},
    ],
    "history": [
        {"title_th": "Sapiens", "title_en": "Sapiens", "author": "Yuval Noah Harari", "rating": 4.5},
        {"title_th": "Guns, Germs, and Steel", "title_en": "Guns, Germs, and Steel", "author": "Jared Diamond", "rating": 4.1},
        {"title_th": "ประวัติศาสตร์ชาติไทย", "title_en": "History of Thailand", "author": "สุจิตต์ วงษ์เทศ", "rating": 4.2},
        {"title_th": "A Short History of Nearly Everything", "title_en": "A Short History of Nearly Everything", "author": "Bill Bryson", "rating": 4.4},
        {"title_th": "The Art of War", "title_en": "The Art of War", "author": "Sun Tzu", "rating": 4.0},
    ],
    "biography": [
        {"title_th": "Steve Jobs", "title_en": "Steve Jobs", "author": "Walter Isaacson", "rating": 4.3},
        {"title_th": "Educated", "title_en": "Educated", "author": "Tara Westover", "rating": 4.5},
        {"title_th": "Becoming", "title_en": "Becoming", "author": "Michelle Obama", "rating": 4.4},
        {"title_th": "Elon Musk", "title_en": "Elon Musk", "author": "Ashlee Vance", "rating": 4.2},
        {"title_th": "ชีวิตดี ๆ ที่ลงตัว", "title_en": "Ikigai", "author": "Héctor García", "rating": 4.3},
    ],
    "tech": [
        {"title_th": "Clean Code", "title_en": "Clean Code", "author": "Robert C. Martin", "rating": 4.4},
        {"title_th": "The Pragmatic Programmer", "title_en": "The Pragmatic Programmer", "author": "David Thomas", "rating": 4.5},
        {"title_th": "Designing Data-Intensive Applications", "title_en": "Designing Data-Intensive Applications", "author": "Martin Kleppmann", "rating": 4.7},
        {"title_th": "Hands-On Machine Learning", "title_en": "Hands-On Machine Learning", "author": "Aurélien Géron", "rating": 4.6},
        {"title_th": "AI 2041", "title_en": "AI 2041", "author": "Kai-Fu Lee", "rating": 4.1},
    ],
    "comics": [
        {"title_th": "วันพีซ", "title_en": "One Piece", "author": "Eiichiro Oda", "rating": 4.9},
        {"title_th": "นารูโตะ", "title_en": "Naruto", "author": "Masashi Kishimoto", "rating": 4.6},
        {"title_th": "ดาบพิฆาตอสูร", "title_en": "Demon Slayer", "author": "Koyoharu Gotouge", "rating": 4.7},
        {"title_th": "สแลมดังก์", "title_en": "Slam Dunk", "author": "Takehiko Inoue", "rating": 4.8},
        {"title_th": "ผ่าพิภพไททัน", "title_en": "Attack on Titan", "author": "Hajime Isayama", "rating": 4.5},
    ],
    "ya": [
        {"title_th": "The Hunger Games", "title_en": "The Hunger Games", "author": "Suzanne Collins", "rating": 4.3},
        {"title_th": "Divergent", "title_en": "Divergent", "author": "Veronica Roth", "rating": 4.1},
        {"title_th": "Percy Jackson", "title_en": "Percy Jackson", "author": "Rick Riordan", "rating": 4.4},
        {"title_th": "The Maze Runner", "title_en": "The Maze Runner", "author": "James Dashner", "rating": 4.0},
        {"title_th": "ด.ญ. เรืองรอง", "title_en": "The Bright Girl", "author": "วินทร์ เลียววาริณ", "rating": 4.2},
    ],
    "food": [
        {"title_th": "Salt Fat Acid Heat", "title_en": "Salt Fat Acid Heat", "author": "Samin Nosrat", "rating": 4.5},
        {"title_th": "ครัวคุณต๋อย", "title_en": "Chef Toy's Kitchen", "author": "หม่อมหลวงขวัญทิพย์", "rating": 4.3},
        {"title_th": "The Food Lab", "title_en": "The Food Lab", "author": "J. Kenji López-Alt", "rating": 4.7},
        {"title_th": "อร่อยทั่วไทย", "title_en": "Delicious Thailand", "author": "ท่องเที่ยวไทย", "rating": 4.1},
        {"title_th": "Mastering the Art of French Cooking", "title_en": "Mastering the Art of French Cooking", "author": "Julia Child", "rating": 4.4},
    ],
    "travel": [
        {"title_th": "เกิดมาเดินทาง", "title_en": "Born to Travel", "author": "นิ้วกลม", "rating": 4.3},
        {"title_th": "Into the Wild", "title_en": "Into the Wild", "author": "Jon Krakauer", "rating": 4.2},
        {"title_th": "Eat Pray Love", "title_en": "Eat Pray Love", "author": "Elizabeth Gilbert", "rating": 3.9},
        {"title_th": "A Walk in the Woods", "title_en": "A Walk in the Woods", "author": "Bill Bryson", "rating": 4.1},
        {"title_th": "ทริปในฝัน", "title_en": "Dream Trip", "author": "สมาคมท่องเที่ยว", "rating": 4.0},
    ],
    "religion": [
        {"title_th": "The Power of Now", "title_en": "The Power of Now", "author": "Eckhart Tolle", "rating": 4.2},
        {"title_th": "พระไตรปิฎกฉบับสำหรับประชาชน", "title_en": "Tipitaka for the Public", "author": "สุชีพ ปุญญานุภาพ", "rating": 4.5},
        {"title_th": "ธรรมะติดปีก", "title_en": "Dhamma with Wings", "author": "ว.วชิรเมธี", "rating": 4.3},
        {"title_th": "Man's Search for Meaning", "title_en": "Man's Search for Meaning", "author": "Viktor E. Frankl", "rating": 4.6},
        {"title_th": "The Alchemist", "title_en": "The Alchemist", "author": "Paulo Coelho", "rating": 4.1},
    ],
    "health": [
        {"title_th": "Why We Sleep", "title_en": "Why We Sleep", "author": "Matthew Walker", "rating": 4.4},
        {"title_th": "The Body", "title_en": "The Body", "author": "Bill Bryson", "rating": 4.3},
        {"title_th": "กินอย่างไรไม่ป่วย", "title_en": "Eat Well Stay Healthy", "author": "นพ.สมเกียรติ", "rating": 4.1},
        {"title_th": "Breath", "title_en": "Breath", "author": "James Nestor", "rating": 4.2},
        {"title_th": "Outlive", "title_en": "Outlive", "author": "Peter Attia", "rating": 4.5},
    ],
    "art": [
        {"title_th": "The War of Art", "title_en": "The War of Art", "author": "Steven Pressfield", "rating": 4.1},
        {"title_th": "Steal Like an Artist", "title_en": "Steal Like an Artist", "author": "Austin Kleon", "rating": 4.2},
        {"title_th": "Ways of Seeing", "title_en": "Ways of Seeing", "author": "John Berger", "rating": 4.0},
        {"title_th": "ศิลปะแห่งการมีชีวิต", "title_en": "The Art of Living", "author": "ธีรยุทธ บุญมี", "rating": 4.3},
        {"title_th": "Big Magic", "title_en": "Big Magic", "author": "Elizabeth Gilbert", "rating": 4.1},
    ],
    "general": [
        {"title_th": "Sapiens", "title_en": "Sapiens", "author": "Yuval Noah Harari", "rating": 4.5},
        {"title_th": "Freakonomics", "title_en": "Freakonomics", "author": "Steven D. Levitt", "rating": 4.0},
        {"title_th": "Factfulness", "title_en": "Factfulness", "author": "Hans Rosling", "rating": 4.4},
        {"title_th": "Cosmos", "title_en": "Cosmos", "author": "Carl Sagan", "rating": 4.5},
        {"title_th": "เรื่องเล่าจากร่างกาย", "title_en": "Stories from the Body", "author": "นพ.สุรชัย", "rating": 4.2},
    ],
    "language": [
        {"title_th": "English Grammar in Use", "title_en": "English Grammar in Use", "author": "Raymond Murphy", "rating": 4.6},
        {"title_th": "ภาษาอังกฤษง่ายนิดเดียว", "title_en": "English is Easy", "author": "ครูดิว", "rating": 4.3},
        {"title_th": "Fluent Forever", "title_en": "Fluent Forever", "author": "Gabriel Wyner", "rating": 4.1},
        {"title_th": "The Elements of Style", "title_en": "The Elements of Style", "author": "Strunk & White", "rating": 4.4},
        {"title_th": "Word Power Made Easy", "title_en": "Word Power Made Easy", "author": "Norman Lewis", "rating": 4.5},
    ],
    "family": [
        {"title_th": "How to Talk So Kids Will Listen", "title_en": "How to Talk So Kids Will Listen", "author": "Adele Faber", "rating": 4.3},
        {"title_th": "The Whole-Brain Child", "title_en": "The Whole-Brain Child", "author": "Daniel J. Siegel", "rating": 4.4},
        {"title_th": "เลี้ยงลูกอย่างไรให้เป็นคนดี", "title_en": "Raise Good Children", "author": "ดร.สุริยเดว", "rating": 4.2},
        {"title_th": "Positive Discipline", "title_en": "Positive Discipline", "author": "Jane Nelsen", "rating": 4.1},
        {"title_th": "วินัยเชิงบวก", "title_en": "Positive Parenting", "author": "Rebecca Eanes", "rating": 4.0},
    ],
    "classics": [
        {"title_th": "1984", "title_en": "1984", "author": "George Orwell", "rating": 4.7},
        {"title_th": "To Kill a Mockingbird", "title_en": "To Kill a Mockingbird", "author": "Harper Lee", "rating": 4.3},
        {"title_th": "Pride and Prejudice", "title_en": "Pride and Prejudice", "author": "Jane Austen", "rating": 4.3},
        {"title_th": "The Great Gatsby", "title_en": "The Great Gatsby", "author": "F. Scott Fitzgerald", "rating": 3.9},
        {"title_th": "สี่แผ่นดิน", "title_en": "Four Reigns", "author": "ม.ร.ว.คึกฤทธิ์ ปราโมช", "rating": 4.6},
    ],
}

print(f"✅ Sample data defined for {len(SAMPLE_BOOKS)} genres")
print(f"📊 Total sample books: {sum(len(v) for v in SAMPLE_BOOKS.values())}")

✅ Sample data defined for 20 genres
📊 Total sample books: 100


## 6. Clean, Structure, and Merge All Data

รวมข้อมูลจาก local scraping, live scraping, และ sample data เข้าด้วยกัน พร้อมสร้าง JSON สำหรับข้อมูลเสริม (FAQ, News, Groups, Discussions, Quotes)

In [6]:
def build_all_books():
    """Merge scraped data with sample data, ensuring all genres have books."""
    all_books = []
    genres_with_data = set()
    
    # 1. Add books from local HTML parsing
    for book in local_books:
        all_books.append(book)
        genres_with_data.add(book["genre"])
    
    # 2. Add books from live scraping
    for book in scraped_books:
        all_books.append(book)
        genres_with_data.add(book["genre"])
    
    # 3. Fill in missing genres with sample data
    for genre_id, sample_list in SAMPLE_BOOKS.items():
        if genre_id not in genres_with_data or len([b for b in all_books if b["genre"] == genre_id]) < 3:
            for sb in sample_list:
                all_books.append({
                    "book_id": f"sample-{genre_id}-{hash(sb['title_en']) % 10000}",
                    "title_th": sb["title_th"],
                    "title_en": sb["title_en"],
                    "author": sb.get("author", ""),
                    "cover_url": "",
                    "genre": genre_id,
                    "rating": sb.get("rating", 4.0),
                    "source": "sample"
                })
    
    return all_books

all_books = build_all_books()

# Summary
from collections import Counter
genre_counts = Counter(b["genre"] for b in all_books)
source_counts = Counter(b["source"] for b in all_books)

print(f"📊 Total books: {len(all_books)}")
print(f"\n📁 By source:")
for src, cnt in source_counts.most_common():
    print(f"   {src}: {cnt}")
print(f"\n📚 By genre:")
for gid, cnt in sorted(genre_counts.items()):
    genre = next((g for g in GENRE_CATEGORIES if g["id"] == gid), None)
    name = genre["th"] if genre else gid
    print(f"   {name}: {cnt}")

📊 Total books: 115

📁 By source:
   sample: 95
   ookbee_local: 20

📚 By genre:
   ศิลปะและการออกแบบ: 5
   ชีวประวัติและบันทึกความทรงจำ: 5
   ธุรกิจ การเงิน และการลงทุน: 5
   วรรณกรรมคลาสสิก: 5
   การ์ตูนและมังงะ: 5
   ครอบครัวและเด็ก: 5
   แฟนตาซี: 5
   อาหารและเครื่องดื่ม: 5
   ความรู้ทั่วไป: 5
   สุขภาพและความงาม: 5
   ประวัติศาสตร์: 5
   ภาษาและการศึกษา: 5
   สืบสวนสอบสวนและระทึกขวัญ: 5
   จิตวิทยาและการพัฒนาตนเอง: 20
   ศาสนาและความเชื่อ: 5
   นิยายโรแมนซ์: 5
   วิทยาศาสตร์: 5
   คอมพิวเตอร์และเทคโนโลยี: 5
   ท่องเที่ยว: 5
   วรรณกรรมเยาวชน: 5


## 7. Save All Data to JSON Files

บันทึกข้อมูลทั้งหมดเป็นไฟล์ JSON ไปที่ `../data/` สำหรับใช้ใน Web App

In [7]:
# Build the complete data structure for the web app
app_data = {
    "genres": GENRE_CATEGORIES,
    "books": all_books,
    "faq": [
        {"q_th": "ใช้เวลานานแค่ไหนในการรับคำแนะนำหนังสือ?", "q_en": "How long does it take to receive book recommendations?",
         "a_th": "ระบบจะเริ่มแนะนำหนังสือทันทีหลังจากคุณเลือกหมวดหมู่ที่ชื่นชอบ และจะยิ่งแม่นยำขึ้นเมื่อคุณให้คะแนนหนังสือมากขึ้น (แนะนำอย่างน้อย 20 เล่ม)",
         "a_en": "The system starts recommending books immediately after you select your favorite genres, and becomes more accurate as you rate more books (recommended at least 20)."},
        {"q_th": "หากได้รับคำแนะนำที่ไม่ตรงใจ ควรทำอย่างไร?", "q_en": "What should I do if I receive irrelevant recommendations?",
         "a_th": "คุณสามารถกดปุ่ม \"ไม่สนใจ\" บนหนังสือที่ไม่ตรงใจ ระบบจะเรียนรู้และปรับปรุงคำแนะนำให้ดีขึ้นในครั้งต่อไป",
         "a_en": "You can click the \"Not Interested\" button on irrelevant books. The system will learn and improve recommendations in the future."},
        {"q_th": "ฉันสามารถตรวจสอบสถานะคำแนะนำหนังสือได้ที่ไหน?", "q_en": "How can I check the status of my book recommendations?",
         "a_th": "ไปที่เมนู \"สำรวจ > แนะนำสำหรับคุณ\" เพื่อดูคำแนะนำล่าสุดของคุณ",
         "a_en": "Go to \"Browse > Recommendations\" to see your latest recommendations."},
        {"q_th": "ฉันสามารถยกเลิกหรือลบหนังสือออกจากคอลเลกชันได้หรือไม่?", "q_en": "Can I remove books from my collection?",
         "a_th": "ได้ คุณสามารถลบหนังสือออกจากชั้นหนังสือได้ตลอดเวลาในหน้า \"หนังสือของฉัน\"",
         "a_en": "Yes, you can remove books from your shelves anytime on the \"My Books\" page."},
        {"q_th": "ติดต่อฝ่ายสนับสนุนได้อย่างไร?", "q_en": "How can I contact customer support?",
         "a_th": "คุณสามารถติดต่อเราได้ที่ team@greedroute.com หรือโทร +66 2 123 4567",
         "a_en": "You can reach us at team@greedroute.com or call +66 2 123 4567"},
        {"q_th": "มีโปรโมชันหรือส่วนลดพิเศษสำหรับผู้ใช้ Greed Route หรือไม่?", "q_en": "Are there special offers for Greed Route users?",
         "a_th": "เรามีโปรโมชันพิเศษเป็นระยะ ๆ ติดตามได้ที่หน้าข่าวสารหรือสมัครรับจดหมายข่าว",
         "a_en": "We have periodic special promotions. Follow our News page or subscribe to our newsletter."},
        {"q_th": "Greed Route มั่นใจในคุณภาพคำแนะนำหนังสือได้อย่างไร?", "q_en": "How does Greed Route ensure recommendation quality?",
         "a_th": "เราใช้อัลกอริทึม Collaborative Filtering และ Content-Based Filtering ผสมผสานกัน พร้อมปรับปรุงจากฟีดแบ็กของผู้ใช้อย่างต่อเนื่อง",
         "a_en": "We use a combination of Collaborative Filtering and Content-Based Filtering algorithms, continuously improved with user feedback."}
    ],
    "news": [
        {"title_th": "10 หนังสือที่ต้องอ่านในปี 2026", "title_en": "10 Must-Read Books of 2026",
         "desc_th": "รวมหนังสือที่ได้รับการคัดเลือกจากผู้เชี่ยวชาญและนักอ่านทั่วโลก",
         "desc_en": "A curated list from experts and readers worldwide", "date": "10 เม.ย. 2569"},
        {"title_th": "บทสัมภาษณ์: ปราบดา หยุ่น กับนวนิยายเรื่องใหม่", "title_en": "Interview: Prabda Yoon on His New Novel",
         "desc_th": "นักเขียนชื่อดังเปิดใจเรื่องแรงบันดาลใจและกระบวนการเขียน",
         "desc_en": "The famous author opens up about inspiration and writing process", "date": "8 เม.ย. 2569"},
        {"title_th": "AI กำลังเปลี่ยนวงการหนังสืออย่างไร?", "title_en": "How AI is Transforming the Book Industry",
         "desc_th": "เจาะลึกว่าปัญญาประดิษฐ์มีบทบาทในการแนะนำหนังสือ การเขียน และการจัดพิมพ์อย่างไร",
         "desc_en": "Exploring how AI plays a role in book recommendations, writing, and publishing", "date": "5 เม.ย. 2569"},
        {"title_th": "รีวิว: \"ชีวิตอันเหลือเชื่อของ...\" โดย ภูมิภาค", "title_en": "Review: \"The Incredible Life of...\" by Phumiphak",
         "desc_th": "หนังสือที่ทำให้คุณหัวเราะและร้องไห้在เวลาเดียวกัน",
         "desc_en": "A book that makes you laugh and cry at the same time", "date": "3 เม.ย. 2569"},
        {"title_th": "งานสัปดาห์หนังสือแห่งชาติ 2569", "title_en": "National Book Week 2026",
         "desc_th": "รวมไฮไลท์และโปรโมชันพิเศษจากงาน",
         "desc_en": "Highlights and special promotions from the event", "date": "1 เม.ย. 2569"}
    ],
    "discussions": [
        {"title_th": "ทำไม Atomic Habits ถึงเปลี่ยนชีวิตคนได้?", "title_en": "Why Can Atomic Habits Change Lives?",
         "group": "Self-Help Lovers", "replies": 45, "date": "วันนี้"},
        {"title_th": "หนังสือแฟนตาซีเรื่องไหนที่ดีที่สุดตลอดกาล?", "title_en": "What Is the Best Fantasy Book of All Time?",
         "group": "Fantasy Readers TH", "replies": 128, "date": "เมื่อวาน"},
        {"title_th": "แนะนำนิยายไทยที่ควรอ่านก่อนตาย", "title_en": "Must-Read Thai Novels Before You Die",
         "group": "Thai Literature", "replies": 67, "date": "2 วันที่แล้ว"},
        {"title_th": "Clean Code vs The Pragmatic Programmer?", "title_en": "Clean Code vs The Pragmatic Programmer?",
         "group": "Dev Bookworms", "replies": 34, "date": "3 วันที่แล้ว"},
        {"title_th": "ใครอ่าน Dune จบแล้วบ้าง? มาพูดคุยกัน!", "title_en": "Who Finished Dune? Let's Discuss!",
         "group": "Sci-Fi Thailand", "replies": 89, "date": "4 วันที่แล้ว"}
    ],
    "quotes": [
        {"text_th": "\"การอ่านหนังสือเล่มดีเล่มหนึ่ง เปรียบเสมือนการสนทนากับจิตใจที่ยิ่งใหญ่ที่สุดของศตวรรษที่ผ่านมา\"",
         "text_en": "\"Reading a great book is like having a conversation with the finest minds of past centuries.\"",
         "author": "René Descartes", "book": "Discourse on the Method", "likes": 1243},
        {"text_th": "\"คนที่ไม่อ่านหนังสือ ไม่ได้มีข้อได้เปรียบเหนือคนที่อ่านไม่ออกเลย\"",
         "text_en": "\"A person who won't read has no advantage over one who can't read.\"",
         "author": "Mark Twain", "book": "", "likes": 987},
        {"text_th": "\"หนังสือเป็นกระจกวิเศษ: หากคนโง่มองเข้าไป คุณจะไม่คาดหวังว่าอัจฉริยะจะมองกลับออกมา\"",
         "text_en": "\"A book is a mirror: if a fool peers in, you can't expect a genius to look back.\"",
         "author": "J.K. Rowling", "book": "", "likes": 856},
        {"text_th": "\"ยิ่งฉันอ่านมากเท่าไร ยิ่งรู้มากเท่านั้น ยิ่งรู้มากเท่าไร ยิ่งไปได้ไกลมากเท่านั้น\"",
         "text_en": "\"The more that you read, the more things you will know. The more that you learn, the more places you'll go.\"",
         "author": "Dr. Seuss", "book": "I Can Read with My Eyes Shut!", "likes": 2145}
    ],
    "groups": [
        {"name_th": "นักอ่านจิตวิทยา", "name_en": "Psychology Readers", "members": 12540, "icon": "🧠"},
        {"name_th": "คนรักนิยาย", "name_en": "Fiction Lovers", "members": 8930, "icon": "📖"},
        {"name_th": "แฟนตาซีไทยแลนด์", "name_en": "Fantasy Thailand", "members": 6720, "icon": "🧙"},
        {"name_th": "Dev Bookworms", "name_en": "Dev Bookworms", "members": 4510, "icon": "💻"},
        {"name_th": "มังงะคลับ", "name_en": "Manga Club", "members": 15200, "icon": "📚"},
        {"name_th": "Sci-Fi Thailand", "name_en": "Sci-Fi Thailand", "members": 3890, "icon": "🚀"},
        {"name_th": "ชมรมนักเขียน", "name_en": "Writers Circle", "members": 2340, "icon": "✍️"},
        {"name_th": "คนรักสุขภาพ", "name_en": "Health & Wellness", "members": 5670, "icon": "💪"}
    ]
}

# Save main data file
output_path = DATA_DIR / "books_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(app_data, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.1f} KB")

# Also save a separate books-only file for easier loading
books_path = DATA_DIR / "books.json"
with open(books_path, "w", encoding="utf-8") as f:
    json.dump(all_books, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {books_path}")

# Save genres
genres_path = DATA_DIR / "genres.json"
with open(genres_path, "w", encoding="utf-8") as f:
    json.dump(GENRE_CATEGORIES, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {genres_path}")

print(f"\n📂 All data files saved to: {DATA_DIR}")

✅ Saved: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data\books_data.json
   File size: 54.6 KB
✅ Saved: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data\books.json
✅ Saved: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data\genres.json

📂 All data files saved to: C:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data


## 8. Verify Output Files

ตรวจสอบไฟล์ที่สร้างขึ้นและแสดงตัวอย่างข้อมูล

In [8]:
# Verify saved files
print("📂 Files in data directory:")
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size
        print(f"   {f.name}: {size/1024:.1f} KB")

# Preview sample data
print("\n📖 Sample books (first 3):")
with open(DATA_DIR / "books_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for book in data["books"][:3]:
    print(f"   {book['title_th']} by {book['author']} ⭐{book['rating']} [{book['genre']}]")

print(f"\n✅ Data ready for web app!")
print(f"   Load in HTML with: fetch('../data/books_data.json')")

📂 Files in data directory:
   books.json: 33.6 KB
   books_data.json: 54.6 KB
   genres.json: 7.2 KB

📖 Sample books (first 3):
   LEAN คิดแบบผู้นำ ทำแบบคนสำเร็จ by  ⭐4.2 [psychology]
   สอนคิดตามแนวทาง Thinking School by  ⭐4.4 [psychology]
   คนแบบไหนอยู่ด้วยแล้วสบายใจ คนแบบไหนอยู่ใกล้แล้วเพลีย by  ⭐4.6 [psychology]

✅ Data ready for web app!
   Load in HTML with: fetch('../data/books_data.json')
